# 13 — Hospital Enrichment → Retrain → Ops

Flow hợp nhất: tải real/enriched data → mô phỏng ba bệnh viện → retrain LightGBM và Logistic Regression → stress-test nhiều seed → inference có data-quality warning. Đây là research/ops simulation, chưa phải clinical deployment.

In [ ]:
!pip install -q gdown scikit-learn lightgbm

In [ ]:
import json
import time
from pathlib import Path

import gdown
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, brier_score_loss, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

RANDOM_STATE = 42
MODEL_SEEDS = [42, 52, 62, 72, 82]
HOSPITAL_TEST_SEEDS = list(range(100, 120))
DOWNLOAD_DIR = Path('/content/hospital_ops_data')
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 100)

## 1. Tải và nhận diện dữ liệu Drive

Notebook tự nhận diện Cleveland gốc, enriched 3.000, synthetic-only 3.000 và locked real test. Các file Drive cần quyền *Anyone with the link — Viewer*.

In [ ]:
DRIVE_IDS = [
 '1YTzUy_RreXqnM5fqMR0hOLZPeXOvYoju',
 '1ezKHfVWmu5kQHBKQllfrr5YEBU4wQJpn',
 '1SOaF95H5QoUi0OkvuLdeAwFskG-x772L',
 '170nGcuSSvitwMgepyEYDjHgpQdAyZ42Q',
 '1sWTkNCcXE3lVcKZmfbjYE9g7gCqIh1da']
FEATURES = ['age','sex','cp','trestbps','chol','fbs','restecg','thalach',
            'exang','oldpeak','slope','ca','thal']
NUMERICAL_FEATURES = ['age','trestbps','chol','thalach','oldpeak']
CATEGORICAL_FEATURES = [c for c in FEATURES if c not in NUMERICAL_FEATURES]
ALL_COLUMNS = FEATURES + ['target']

paths = []
for index, file_id in enumerate(DRIVE_IDS, 1):
    result = gdown.download(id=file_id, output=str(DOWNLOAD_DIR/f'file_{index}'),
                            quiet=False, fuzzy=True)
    if result is None: raise RuntimeError(f'Khong tai duoc Drive file {index}')
    paths.append(Path(result))

datasets = {}
for path in paths:
    try: candidate = pd.read_csv(path)
    except Exception: continue
    if set(ALL_COLUMNS).issubset(candidate.columns):
        if len(candidate)==3000 and 'data_origin' in candidate.columns:
            origins = set(candidate.data_origin.astype(str).unique())
            if 'real_train' in origins:
                datasets['enriched'] = candidate
            else:
                datasets['synthetic_demo'] = candidate
        elif len(candidate)<100: datasets['locked_test'] = candidate
    else:
        original = pd.read_csv(path,header=None,names=FEATURES+['target_original'],na_values=['?'])
        if len(original)==303:
            original['target']=(pd.to_numeric(original.target_original,errors='coerce')>0).astype(int)
            datasets['original']=original[ALL_COLUMNS]
required={'original','enriched','synthetic_demo','locked_test'}
if required-set(datasets): raise ValueError(f'Missing datasets: {required-set(datasets)}')
for name,data in datasets.items(): print(name,data.shape)

## 2. Ba hospital profile

Các tỷ lệ dưới đây là giả định nghiên cứu, không phải thống kê của bệnh viện thật. Khi có aggregate statistics thực tế, thay trực tiếp các profile này.

In [ ]:
HOSPITAL_PROFILES = {
 'H01_cardiac_center': {
   'missing':.02,'outlier':.005,'invalid_code':.002,'rounding':.03,
   'age_shift':5,'bp_shift':5,'shift_probability':.70,'ca_thal_missing':.03},
 'H02_general_hospital': {
   'missing':.08,'outlier':.02,'invalid_code':.01,'rounding':.10,
   'age_shift':1,'bp_shift':2,'shift_probability':.40,'ca_thal_missing':.12},
 'H03_outpatient_clinic': {
   'missing':.15,'outlier':.04,'invalid_code':.02,'rounding':.20,
   'age_shift':-3,'bp_shift':0,'shift_probability':.60,'ca_thal_missing':.35},
}
display(pd.DataFrame(HOSPITAL_PROFILES).T)

In [ ]:
def apply_hospital_profile(frame, hospital_id, seed):
    cfg=HOSPITAL_PROFILES[hospital_id]; result=frame.copy().reset_index(drop=True)
    rng=np.random.default_rng(seed)
    result[FEATURES]=result[FEATURES].mask(rng.random((len(result),len(FEATURES)))<cfg['missing'])
    for column in ['ca','thal']:
        result.loc[rng.random(len(result))<cfg['ca_thal_missing'],column]=np.nan
    for column in NUMERICAL_FEATURES:
        mask=rng.random(len(result))<cfg['outlier']
        result.loc[mask,column]*=rng.choice([.1,.5,1.5,10.0],size=mask.sum())
    for column,step in {'trestbps':5,'chol':10,'thalach':5,'oldpeak':.5}.items():
        mask=rng.random(len(result))<cfg['rounding']
        result.loc[mask,column]=(result.loc[mask,column]/step).round()*step
    for column in CATEGORICAL_FEATURES:
        result.loc[rng.random(len(result))<cfg['invalid_code'],column]=99
    shifted=rng.random(len(result))<cfg['shift_probability']
    result.loc[shifted,'age']+=cfg['age_shift']
    result.loc[shifted,'trestbps']+=cfg['bp_shift']
    result['hospital_id']=hospital_id
    result['noise_level']=pd.cut(
        result[FEATURES].isna().sum(axis=1),bins=[-1,0,2,99],labels=['low','medium','high'])
    return result


## 3. Tạo enriched hospital training set

Giữ 242 real-train rows nguyên bản. Chỉ phân phối/corrupt các synthetic rows qua ba hospital profile để tránh phá tín hiệu thật.

In [ ]:
enriched_source=datasets['enriched'].copy()
real_train=enriched_source[enriched_source.data_origin=='real_train'][ALL_COLUMNS].copy()
synthetic_source=enriched_source[enriched_source.data_origin!='real_train'][ALL_COLUMNS].copy()
locked_test=datasets['locked_test'][ALL_COLUMNS].copy()
synthetic_demo=datasets['synthetic_demo'][ALL_COLUMNS].copy()
for frame in [real_train,synthetic_source,locked_test,synthetic_demo]:
    for column in ALL_COLUMNS: frame[column]=pd.to_numeric(frame[column],errors='coerce')

assignment_rng=np.random.default_rng(RANDOM_STATE)
assignments=assignment_rng.choice(list(HOSPITAL_PROFILES),len(synthetic_source),p=[.35,.40,.25])
hospital_parts=[]
for idx,hospital_id in enumerate(HOSPITAL_PROFILES):
    part=synthetic_source.loc[assignments==hospital_id].copy()
    hospital_parts.append(apply_hospital_profile(part,hospital_id,RANDOM_STATE+idx))
real_part=real_train.copy(); real_part['hospital_id']='REAL_CLEVELAND'; real_part['noise_level']='observed'
enriched_hospital=pd.concat([real_part,*hospital_parts],ignore_index=True).sample(
    frac=1,random_state=RANDOM_STATE).reset_index(drop=True)
assert len(enriched_hospital)==3000
training_sets={'real_only':real_train,'enriched_hospital_3000':enriched_hospital}
display(enriched_hospital.groupby(['hospital_id','noise_level'],observed=True).size().rename('rows'))
print('Missing enriched cells:',int(enriched_hospital[FEATURES].isna().sum().sum()))

## 4. Retrain model cũ và stress-test nhiều seed

So sánh LightGBM và Logistic Regression, mỗi model train trên real-only hoặc enriched-hospital. Test gồm clean real và ba hospital profile qua 20 corruption seed.

In [ ]:
def make_preprocessor():
    numeric=Pipeline([('imputer',SimpleImputer(strategy='median',add_indicator=True)),
                      ('scaler',MinMaxScaler())])
    categorical=Pipeline([('imputer',SimpleImputer(strategy='most_frequent',add_indicator=True)),
                          ('encoder',OneHotEncoder(handle_unknown='ignore'))])
    return ColumnTransformer([('num',numeric,NUMERICAL_FEATURES),
                              ('cat',categorical,CATEGORICAL_FEATURES)])

def make_models(seed):
    return {
      'Logistic Regression':LogisticRegression(max_iter=1500,random_state=seed),
      'LightGBM':LGBMClassifier(n_estimators=300,num_leaves=15,learning_rate=.03,
          min_child_samples=15,random_state=seed,verbosity=-1,n_jobs=-1)}

records=[]
for train_name,training in training_sets.items():
  for model_seed in MODEL_SEEDS:
    for model_name,classifier in make_models(model_seed).items():
      pipeline=Pipeline([('preprocessor',make_preprocessor()),('classifier',classifier)])
      started=time.perf_counter(); pipeline.fit(training[FEATURES],training.target.astype(int))
      fit_seconds=time.perf_counter()-started
      test_variants=[('clean',0,locked_test)]
      for hospital_id in HOSPITAL_PROFILES:
        for noise_seed in HOSPITAL_TEST_SEEDS:
          test_variants.append((hospital_id,noise_seed,
              apply_hospital_profile(locked_test,hospital_id,noise_seed)))
      for profile,noise_seed,testing in test_variants:
        prob=pipeline.predict_proba(testing[FEATURES])[:,1]; pred=(prob>=.5).astype(int)
        tn,fp,fn,tp=confusion_matrix(testing.target,pred,labels=[0,1]).ravel()
        records.append({'train_set':train_name,'model':model_name,'model_seed':model_seed,
          'test_profile':profile,'noise_seed':noise_seed,
          'accuracy':accuracy_score(testing.target,pred),
          'precision':precision_score(testing.target,pred,zero_division=0),
          'recall':recall_score(testing.target,pred,zero_division=0),
          'specificity':tn/(tn+fp) if tn+fp else np.nan,
          'f1':f1_score(testing.target,pred,zero_division=0),
          'roc_auc':roc_auc_score(testing.target,prob),
          'brier':brier_score_loss(testing.target,prob),
          'false_negatives':int(fn),'fit_seconds':fit_seconds})
results=pd.DataFrame(records)
print('Evaluation rows:',len(results)); display(results.head())

In [ ]:
summary=results.groupby(['train_set','model','test_profile']).agg(
  roc_auc_mean=('roc_auc','mean'),roc_auc_std=('roc_auc','std'),
  recall_mean=('recall','mean'),recall_std=('recall','std'),recall_min=('recall','min'),
  f1_mean=('f1','mean'),brier_mean=('brier','mean'),
  false_negatives_mean=('false_negatives','mean')).reset_index()
display(summary.sort_values(['test_profile','roc_auc_mean'],ascending=[True,False]).round(4))

robust=summary[summary.test_profile!='clean'].groupby(['train_set','model']).agg(
  hospital_auc_mean=('roc_auc_mean','mean'),hospital_auc_worst=('roc_auc_mean','min'),
  hospital_recall_mean=('recall_mean','mean'),hospital_recall_worst=('recall_min','min'),
  hospital_brier_mean=('brier_mean','mean'),
  false_negatives_mean=('false_negatives_mean','mean')).reset_index().sort_values(
    ['hospital_auc_mean','hospital_recall_mean','hospital_brier_mean'],ascending=[False,False,True])
display(robust.round(4))

plt.figure(figsize=(14,6)); sns.barplot(data=summary,x='test_profile',y='roc_auc_mean',
    hue=summary['train_set']+' | '+summary['model'])
plt.ylim(.5,1); plt.xticks(rotation=15); plt.title('Hospital robustness comparison'); plt.show()

## 5. Fit deployment candidate và Ops inference

Candidate được chọn theo hospital AUC trung bình trong lab. Đây là exploratory selection; production model vẫn cần external validation. Hàm inference trả thêm missing/outlier/invalid-code warnings.

In [ ]:
candidate=robust.iloc[0]
candidate_train=candidate.train_set; candidate_model=candidate.model
final_classifier=make_models(RANDOM_STATE)[candidate_model]
final_pipeline=Pipeline([('preprocessor',make_preprocessor()),('classifier',final_classifier)])
final_pipeline.fit(training_sets[candidate_train][FEATURES],
                   training_sets[candidate_train].target.astype(int))
print('OPS CANDIDATE:',candidate_train,'+',candidate_model)

REAL_BOUNDS={c:(float(real_train[c].quantile(.01)),float(real_train[c].quantile(.99)))
             for c in NUMERICAL_FEATURES}
VALID_CATEGORIES={c:set(real_train[c].dropna().unique()) for c in CATEGORICAL_FEATURES}

def predict_patient(patient, threshold=.5):
    frame=pd.DataFrame([patient]) if isinstance(patient,dict) else patient.copy()
    missing_columns=[c for c in FEATURES if c not in frame.columns]
    if missing_columns: raise ValueError(f'Missing required columns: {missing_columns}')
    frame=frame[FEATURES].copy()
    missing_features=[c for c in FEATURES if frame[c].isna().any()]
    outliers=[]
    for c,(low,high) in REAL_BOUNDS.items():
        if ((frame[c]<low)|(frame[c]>high)).fillna(False).any(): outliers.append(c)
    invalid_codes=[]
    for c,allowed in VALID_CATEGORIES.items():
        if (~frame[c].isin(allowed)&frame[c].notna()).any(): invalid_codes.append(c)
    probability=final_pipeline.predict_proba(frame)[:,1]
    quality_issues=len(missing_features)+len(outliers)+len(invalid_codes)
    quality='high' if quality_issues==0 else ('medium' if quality_issues<=2 else 'low')
    return pd.DataFrame({'prediction':(probability>=threshold).astype(int),
      'probability':probability.round(4),'threshold':threshold,'data_quality':quality,
      'missing_features':[missing_features]*len(frame),
      'outlier_features':[outliers]*len(frame),
      'invalid_code_features':[invalid_codes]*len(frame)})

patient_example={
 'age':55,'sex':1,'cp':2,'trestbps':130,'chol':250,'fbs':0,'restecg':1,
 'thalach':150,'exang':0,'oldpeak':2.0,'slope':2,'ca':np.nan,'thal':np.nan}
display(predict_patient(patient_example))

In [ ]:
# Batch ops trên 3.000 hồ sơ demo; không dùng kết quả này làm clinical metric.
batch_probability=final_pipeline.predict_proba(synthetic_demo[FEATURES])[:,1]
batch_summary={
 'rows':len(synthetic_demo),'predicted_positive_rate':float((batch_probability>=.5).mean()),
 'probability_mean':float(batch_probability.mean()),
 'probability_p05':float(np.quantile(batch_probability,.05)),
 'probability_p95':float(np.quantile(batch_probability,.95)),
 'missing_cells':int(synthetic_demo[FEATURES].isna().sum().sum())}
display(pd.Series(batch_summary,name='batch_ops_summary'))

## 6. Quy tắc quyết định

Chỉ chọn enriched model nếu clean AUC không giảm quá 0.01, hospital AUC/Recall cải thiện qua nhiều seed và Brier không xấu rõ rệt. Nếu không đạt, giữ `real-only + LightGBM`. `hospital_id`, `noise_level` và `data_origin` không được dùng làm predictor. Notebook không xuất model; bước xuất artifact chỉ thực hiện sau external validation và chốt pipeline.